In [1]:
# Load packages
import os

# Set workspace environment variables
os.environ["WORKSPACE_BUCKET"] = "gs://rw-migration-aou-rw-9f879fc4"
os.environ["ARTIFACT_REGISTRY_DOCKER_REPO"] = (
    "us-central1-docker.pkg.dev/all-of-us-rw-prod/aou-rw-gar-remote-repo-docker-prod"
)

# Load the remaining workspace environment settings
%run Setting_Env_Variables.ipynb

Successfully set WORKSPACE_CDR to: wb-silky-artichoke-2408.C2024Q3R9
Found bucket: id=dataproc-staging-aou-rw-9f879fc4, bucketName=dataproc-staging-wb-fast-radish-6415
Found bucket: id=dataproc-temp-aou-rw-9f879fc4, bucketName=dataproc-temp-wb-fast-radish-6415
Found bucket: id=rw-migration-aou-rw-9f879fc4, bucketName=rw-migration-aou-rw-9f879fc4
Found bucket: id=vwb-aou-datasets-controlled-v8r9, bucketName=vwb-aou-datasets-controlled

Variables extracted:
GOOGLE_CLOUD_PROJECT: wb-fast-radish-6415
WORKSPACE_BUCKET: gs://rw-migration-aou-rw-9f879fc4
WORKSPACE_TEMP_BUCKET: NOT FOUND
WORKSPACE_CDR: wb-silky-artichoke-2408.C2024Q3R9

✅ Saved to /home/dataproc/.bashrc
C2024Q3R9 BQ_DATASET
JupyterLab_Spark_cluster_for_AoU_Spark_20261102 DATAPROC_CLUSTER
dataproc-staging-aou-rw-9f879fc4 GCS_BUCKET
dataproc-temp-aou-rw-9f879fc4 GCS_BUCKET
rw-migration-aou-rw-9f879fc4 GCS_BUCKET
vwb-aou-datasets-controlled-v8r9 GCS_BUCKET
GOOGLE_CLOUD_PROJECT = wb-fast-radish-6415
WORKSPACE_BUCKET = gs://rw-migr

# COMMAND 

In [2]:
%%writefile gds2bgen.R
#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Load packages 
library(gds2bgen)
library(SeqArray)

# Read command-line arguments supplied to the R script
args <- commandArgs(trailingOnly = TRUE)

if (length(args) != 3) {
    stop("Usage: Rscript gds2bgen.R <input.bgen> <output.gds> <n.cpus")
}

# Require exactly three arguments:
#   <input.bgen>  = input BGEN file
#   <output.gds>  = output GDS file
#   <n.cpus>      = number of CPUs/threads to use
# Assign command-line arguments to descriptive variables
bgen_fn <- args[1]
gds_fn  <- args[2]
cpu <- as.integer(args[3])

# Print the input/output files and CPU count for logging
cat("BGEN:", bgen_fn, "\n")
cat("GDS:", gds_fn, "\n")
cat("CPU:", cpu, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################################

seqBGEN_Info() # bgen library version
## "bgen_lib_v1.1.8"

# Display information about the input BGEN file
# This is useful for checking that the BGEN file can be read successfully
seqBGEN_Info(bgen_fn)

# Convert the BGEN file into a GDS file
seqBGEN2GDS(
    bgen_fn,                          # Input BGEN file
    gds_fn,                           # Output GDS file
    storage.option = "LZMA_RA",       # LZMA compression with random access  
    float.type = "packed8",           # Store floating-point values in packed 8-byte format
    geno = TRUE,                      # Store genotype data
    dosage = FALSE,                    # Store dosage data
    prob = FALSE,                     # Do not store genotype probabilities
    parallel = cpu                    # Number of CPUs to use
)


###########################################################################
# VERIFY THE CONVERTED GDS FILE
###########################################################################

# Open the newly created GDS file for reading
gds <- seqOpen(gds_fn)

# Extract basic variant information from the GDS
CHR <- seqGetData(gds, "chromosome")
POS <- seqGetData(gds, "position")
REF <- seqGetData(gds, "$ref")
ALT <- seqGetData(gds, "$alt")

# Combine the variant information into a data frame
# This provides a convenient way to inspect the first variants
test <- data.frame(
    CHR = CHR,
    POS = POS,
    REF = REF,
    ALT = ALT
)

# Print the first 10 variants to verify the conversion
print(head(test, 10), row.names = FALSE)

# Close the GDS file
seqClose(gds)

###########################################################################
# REOPEN GDS FOR MODIFICATION
###########################################################################

# Reopen the GDS file with write access
genofile <- seqOpen(gds_fn, readonly = FALSE)

# Print the GDS structure so that its contents can be inspected
print(genofile)

# Extract variant positions and convert them to integers
position <- as.integer(seqGetData(genofile, "position"))

# Print the GDS structure again
print(genofile)

###########################################################################
# ADD QC ANNOTATION
###########################################################################

# Locate the GDS annotation/info node
# This is where the new QC_label variable will be added
Anno.folder <- index.gdsn(genofile, "annotation/info")

# Print the GDS structure again
print(genofile)

# Add a new QC_label annotation
#
# Every variant is assigned the label "PASS".
# factor() stores the values as a categorical variable.
#
# length(position) determines the number of variants, ensuring that
# one QC label is created for every variant.
add.gdsn(Anno.folder, 
         "QC_label", 
         val=factor(rep("PASS", length(position))), 
         compress="LZMA_ra", 
         closezip=TRUE)

# Print the GDS structure again
print(genofile)

###########################################################################
# VERIFY VARIANT INFORMATION AFTER MODIFICATION
###########################################################################

# Reload the variant information from the modified GDS
CHR <- seqGetData(genofile, "chromosome")
POS <- seqGetData(genofile, "position")
REF <- seqGetData(genofile, "$ref")
ALT <- seqGetData(genofile, "$alt")

# Print the GDS structure again
print(genofile)

# Recreate the variant information table
test <- data.frame(
    CHR = CHR,
    POS = POS,
    REF = REF,
    ALT = ALT
)

# Print the GDS structure again
print(genofile)

# Print the first 10 variants to verify the conversion
print(head(test, 10), row.names = FALSE)

# Close the modified GDS file
seqClose(genofile)

Overwriting gds2bgen.R


In [3]:
# Save script to workspace bucket 
!gcloud storage cp gds2bgen.R ${WORKSPACE_BUCKET}/data/scripts/

Copying file://gds2bgen.R to gs://rw-migration-aou-rw-9f879fc4/data/scripts/gds2bgen.R
  Completed files 1/1 | 4.6kiB/4.6kiB                                          


In [4]:
# Check script in workspace bucket 
!gcloud storage cat ${WORKSPACE_BUCKET}/data/scripts/gds2bgen.R 

#!/usr/bin/env Rscript

##########################################################################
# INPUT
##########################################################################

# Load packages 
library(gds2bgen)
library(SeqArray)

# Read command-line arguments supplied to the R script
args <- commandArgs(trailingOnly = TRUE)

if (length(args) != 3) {
    stop("Usage: Rscript gds2bgen.R <input.bgen> <output.gds> <n.cpus")
}

# Require exactly three arguments:
#   <input.bgen>  = input BGEN file
#   <output.gds>  = output GDS file
#   <n.cpus>      = number of CPUs/threads to use
# Assign command-line arguments to descriptive variables
bgen_fn <- args[1]
gds_fn  <- args[2]
cpu <- as.integer(args[3])

# Print the input/output files and CPU count for logging
cat("BGEN:", bgen_fn, "\n")
cat("GDS:", gds_fn, "\n")
cat("CPU:", cpu, "\n")

###########################################################################
# MAIN FUNCTION
###########################################################

# SUBMIT COMMAND 

In [5]:
import subprocess
import pandas as pd

# Create dataframe of size of exome plink files in GB 
chromosome = []
sizes = [] 

# Loop for all chromosome 
# for chrom in range(1, 23):
for chrom in [2, 4, 15, 17]:
    
    # Extract size of genomic plink bed file in GB for each chromosome 
    command = [
        "gcloud", "storage", "ls",
        "-l",
        f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/chr{chrom}.bgen", 
        f"--billing-project={os.environ['GOOGLE_PROJECT']}"
    ]
    
    result = subprocess.run(command, capture_output=True, text=True)

    # Extract genomic size from string 
    lines = result.stdout.splitlines()
    line = lines[0].split()
    
    # Create list genomic size 
    sizes.append(line[0])
    chromosome.append(chrom)
    
# Deposit genomic file size into data frame 
genomic_size = pd.DataFrame({
    "chromosome": chromosome,
    "size": sizes
})

# Format genomic file size 
genomic_size['chromosome'] = genomic_size['chromosome'].astype('int')
genomic_size['size'] = (genomic_size['size'].astype('int') / 1e9) 

print(genomic_size)


   chromosome      size
0           2  4.259703
1           4  2.506960
2          15  2.273354
3          17  3.442683


In [ ]:
import shlex

# Logging directory
LOGGING = f"{os.environ['WORKSPACE_BUCKET']}/data/logging"

# Dsub script
SCRIPT = "gds2bgen.R"
PATH = f"{os.environ['WORKSPACE_BUCKET']}/data/scripts/{SCRIPT}"

MAX_DISK_SIZE = 10 # Set maximum disk; adjust for acaf or exome accordingly
EXTRA_DISK_SIZE = 5 # Set additional disk size for each chromosome 

# Define number of CPUs
CPU = 4

# Loop to submit dsub command
# for CHROM in range(1,23):
for CHROM in [17]:
    
    print(f"Submitting dsub command for chromosome {CHROM}")

    DISK_SIZE = (round(float(genomic_size.loc[genomic_size["chromosome"] == CHROM, "size"].iloc[0])) + EXTRA_DISK_SIZE)

    print(f"Requesting disk size: {DISK_SIZE}")

    if DISK_SIZE > MAX_DISK_SIZE:
        print(f"Disk too large for chromosome {CHROM}")
        break

    if DISK_SIZE > 1000:
        print(f"Disk too large for chromosome {CHROM}")
        break

    # BGEN path
    BGEN = (f"{os.environ['WORKSPACE_BUCKET']}/data/staar/bgen/chr{CHROM}.bgen")
    # GDS path 
    GDS = (f"{os.environ['WORKSPACE_BUCKET']}/data/staar/gds/chr{CHROM}.gds")

    # R script command
    R_COMMAND = 'Rscript "$SCRIPT" "$BGEN" "$GDS" "$CPU"'

    # Submission command 
    command = [
        "aou_dsub",
            "--image", f"{os.environ['ARTIFACT_REGISTRY_DOCKER_REPO']}/vieno99/gds2bgen:v1.0",
            "--boot-disk-size", "5",
            "--disk-size", str(DISK_SIZE),
            "--machine-type", "n1-standard-4",
            "--logging", LOGGING,
            "--input", f"SCRIPT={PATH}",
            "--input", f"BGEN={BGEN}",
            "--output", f"GDS={GDS}",
            "--env", f"CPU={CPU}",
            "--command", R_COMMAND,
    ]

    # Run in same shell as source command
    dsub_command = subprocess.run(
        f"source ~/aou_dsub.bash && {shlex.join(command)}",
        shell=True,
        executable="/bin/bash",
        capture_output=True,
        text=True
    )

    print(dsub_command.stdout)
    print(dsub_command.stderr)

Submitting dsub command for chromosome 17
Requesting disk size: 8
rscript--vienose--261103-202643-05

Job properties:
  job-id: rscript--vienose--261103-202643-05
  job-name: rscript
  user-id: vienose
Provider internal-id (operation): projects/wb-fast-radish-6415/locations/us-central1/jobs/rscript--vienose--261103-202643-05-0-0
Launched job-id: rscript--vienose--261103-202643-05
To check the status, run:
  dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261103-202643-05' --users 'vienose' --status '*'
To cancel the job, run:
  ddel --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261103-202643-05' --users 'vienose'



In [1]:
!dstat --provider google-batch --project wb-fast-radish-6415 --location us-central1 --jobs 'rscript--vienose--261103-202643-05' --users 'vienose' --status '*'

Job Name    Status    Last Update
----------  --------  --------------
rscript     SUCCESS   11-04 00:22:06



In [46]:
!gcloud storage ls gs://rw-migration-aou-rw-9f879fc4/data/logging/ | grep 'rscript--vienose--261103-202643-05'

gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261103-202643-05-stderr.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261103-202643-05-stdout.log
gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261103-202643-05.log


In [2]:
!gcloud storage cat gs://rw-migration-aou-rw-9f879fc4/data/logging/rscript--vienose--261103-202643-05-stdout.log

BGEN: /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/staar/bgen/chr17.bgen 
GDS: /mnt/data/output/gs/rw-migration-aou-rw-9f879fc4/data/staar/gds/chr17.gds 
CPU: 4 
[1] "bgen_lib_v1.1.8"
File: /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/staar/bgen/chr17.bgen
# of samples: 218625
# of variants: 1260020
Compression method: zlib
Layout version: v1.2
Unphased: TRUE
# of bits: 16
Ploidy: 2
Sample id: 1000004_1000004, 1000033_1000033, 1000059_1000059, 1000061_1000061, ...
##< 2026-11-03 20:29:23
BGEN Import:
    file (3.2G):
        /mnt/data/input/gs/rw-migration-aou-rw-9f879fc4/data/staar/bgen/chr17.bgen
    # of samples: 218625
    # of variants: 1260020
    bgen compression method: zlib
    layout version: v1.2
    unphased: TRUE
    # of bits: 16
    ploidy: 2
    sample id: 1000004_1000004, 1000033_1000033, 1000059_1000059, 1000061_1000061, ...
Output:
    /mnt/data/output/gs/rw-migration-aou-rw-9f879fc4/data/staar/gds/chr17.gds
    saving genotypes [GT]: TRUE
    saving 